# Common setup for block AUC notebooks

This notebook is meant to be run by the test notebooks with `%run ./00_block_auc_common.ipynb`.

It defines the HIV feature matrix, the paper-facing immune blocks, and reusable RF AUC helpers. It intentionally does not write CSVs or figures to disk.

In [ ]:
from pathlib import Path
import sys
import os
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
N_REPEATS = 50
N_ESTIMATORS = 200
TEST_SIZE = 0.30

CWD = Path.cwd().resolve()
if CWD.name == 'notebooks':
    NOTEBOOK_DIR = CWD
    DS_ROOT = NOTEBOOK_DIR.parent
    REPO_ROOT = DS_ROOT.parent
elif (CWD / 'data_synthesis').exists():
    REPO_ROOT = CWD
    DS_ROOT = REPO_ROOT / 'data_synthesis'
    NOTEBOOK_DIR = DS_ROOT / 'notebooks'
else:
    DS_ROOT = CWD
    NOTEBOOK_DIR = DS_ROOT / 'notebooks'
    REPO_ROOT = DS_ROOT.parent

for p in [DS_ROOT, DS_ROOT / 'src']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from loaders import load_HIV

hiv = load_HIV()
X = np.asarray(hiv['X'], dtype=np.float64)
y = np.asarray(hiv['y']).ravel().astype(int)
feature_names = list(hiv['feature_names'])
feature_index = {name: i for i, name in enumerate(feature_names)}

n0 = int((y == 0).sum())
n1 = int((y == 1).sum())


In [ ]:
def cols_containing(*tokens):
    return [c for c in feature_names if all(t in c for t in tokens)]

BLOCKS = {
    # Serum IgG is the repeated blood IgG Spike/RBD trajectory columns only.
    # Production/decay is kept separate so dynamic summaries can be tested directly.
    'serum IgG': cols_containing('blood_IgGspike') + cols_containing('blood_IgGRBD'),
    'saliva IgA': [c for c in feature_names if 'Saliva_IgA' in c],
    'saliva IgG': [c for c in feature_names if 'Saliva_IgG' in c],
    'cytokines': [c for c in ['V8_IFNg', 'V9_IFNg', 'V8_IL2', 'V9_IL2', 'V9Dual', 'V8Dual', 'IFNG_production', 'Il2_production'] if c in feature_names],
    'ACE2': [c for c in ['V7_ACE2', 'V8_ACE2', 'V8b_ACE2', 'V9_ACE2'] if c in feature_names],
    'CD4/CD8 + neutrophils': [c for c in ['RATIO_CD4CD8', 'V8Neut', 'V9Neut'] if c in feature_names],
    'production/decay': [c for c in feature_names if ('Production' in c or 'Decay' in c)],
}

PRIMARY_BLOCK_ONLY = ['serum IgG', 'saliva IgA', 'cytokines', 'ACE2', 'production/decay']
PRIMARY_BLOCK_DROP = ['serum IgG', 'production/decay', 'ACE2']

BLOCK_COLORS = {
    'serum IgG': '#9E3934',
    'saliva IgA': '#3675B8',
    'saliva IgG': '#7E9EB2',
    'cytokines': '#F6B541',
    'ACE2': '#94C652',
    'CD4/CD8 + neutrophils': '#757476',
    'production/decay': '#DC3A7A',
}

def feature_indices(cols):
    return np.asarray([feature_index[c] for c in cols if c in feature_index], dtype=int)


def block_features(block_name):
    return BLOCKS[block_name]


def block_indices(block_name):
    return feature_indices(block_features(block_name))


def repeated_hiv_auc(feature_idx, repeats=N_REPEATS, seed=SEED):
    """Repeated RF AUC for HIV status using only feature_idx."""
    if len(feature_idx) == 0:
        return np.repeat(0.5, repeats)
    vals = []
    X_sub = X[:, feature_idx]
    for r in range(repeats):
        X_train, X_test, y_train, y_test = train_test_split(
            X_sub, y, test_size=TEST_SIZE, stratify=y, random_state=seed + r
        )
        rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=seed + r, n_jobs=-1)
        rf.fit(X_train, y_train)
        vals.append(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))
    return np.asarray(vals)


def summarize_auc(vals):
    return {
        'mean': float(np.mean(vals)),
        'sd': float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        'ci_low': float(np.mean(vals) - 1.96 * (np.std(vals, ddof=1) / np.sqrt(len(vals)))) if len(vals) > 1 else float(np.mean(vals)),
        'ci_high': float(np.mean(vals) + 1.96 * (np.std(vals, ddof=1) / np.sqrt(len(vals)))) if len(vals) > 1 else float(np.mean(vals)),
    }


def evaluate_block_only(block_names):
    rows = []
    for block in block_names:
        idx = block_indices(block)
        vals = repeated_hiv_auc(idx)
        s = summarize_auc(vals)
        rows.append({
            'block': block,
            'n_features': len(idx),
            'auc_mean': s['mean'],
            'auc_sd': s['sd'],
            'auc_ci_low': s['ci_low'],
            'auc_ci_high': s['ci_high'],
        })
    return pd.DataFrame(rows)


def evaluate_block_drop(block_names):
    all_idx = set(range(X.shape[1]))
    full_vals = repeated_hiv_auc(np.asarray(sorted(all_idx), dtype=int))
    full = summarize_auc(full_vals)
    rows = [{'block': 'all features', 'n_features': X.shape[1], 'auc_mean': full['mean'], 'auc_sd': full['sd'], 'auc_ci_low': full['ci_low'], 'auc_ci_high': full['ci_high'], 'delta_from_full': 0.0}]
    for block in block_names:
        drop_idx = set(block_indices(block))
        keep_idx = np.asarray(sorted(all_idx - drop_idx), dtype=int)
        vals = repeated_hiv_auc(keep_idx)
        s = summarize_auc(vals)
        rows.append({
            'block': f'all except {block}',
            'dropped_block': block,
            'n_features': len(keep_idx),
            'auc_mean': s['mean'],
            'auc_sd': s['sd'],
            'auc_ci_low': s['ci_low'],
            'auc_ci_high': s['ci_high'],
            'delta_from_full': s['mean'] - full['mean'],
        })
    return pd.DataFrame(rows), full


In [ ]:
def plot_block_definitions(block_names, title='Feature blocks used in this test'):
    counts = [len(block_features(b)) for b in block_names]
    colors = [BLOCK_COLORS.get(b.replace('all except ', ''), '#777777') for b in block_names]
    fig, ax = plt.subplots(figsize=(8.5, 0.55 * len(block_names) + 1.2))
    y_pos = np.arange(len(block_names))
    ax.barh(y_pos, counts, color=colors, alpha=0.9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(block_names)
    ax.invert_yaxis()
    ax.set_xlabel('Number of features')
    ax.set_title(title)
    for y_i, n in zip(y_pos, counts):
        ax.text(n + 0.2, y_i, str(n), va='center')
    ax.spines[['top', 'right']].set_visible(False)
    plt.show()


def show_block_members(block_names):
    lines = []
    for b in block_names:
        wrapped = ', '.join(block_features(b))
        lines.append(f'**{b}** ({len(block_features(b))}): {wrapped}')
    display(Markdown('\n\n'.join(lines)))


def plot_auc_bars(df, title, xlabel='RF AUC for HIV status', xlim=(0.45, 1.02), baseline=None):
    plot_df = df.copy()
    plot_df = plot_df.sort_values('auc_mean', ascending=True)
    labels = plot_df['block'].tolist()
    colors = []
    for label in labels:
        key = label.replace('all except ', '')
        colors.append(BLOCK_COLORS.get(key, '#555555'))
    fig, ax = plt.subplots(figsize=(9, 0.58 * len(plot_df) + 1.5))
    y_pos = np.arange(len(plot_df))
    xerr = np.vstack([
        plot_df['auc_mean'] - plot_df['auc_ci_low'],
        plot_df['auc_ci_high'] - plot_df['auc_mean'],
    ])
    ax.barh(y_pos, plot_df['auc_mean'], xerr=xerr, color=colors, alpha=0.9, capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlim(*xlim)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='chance')
    if baseline is not None:
        ax.axvline(baseline, color='#222222', linestyle=':', linewidth=2, label='all features')
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.spines[['top', 'right']].set_visible(False)
    if baseline is not None:
        ax.legend(frameon=False, loc='lower right')
    for y_i, (_, row) in enumerate(plot_df.iterrows()):
        ax.text(row['auc_mean'] + 0.01, y_i, f"{row['auc_mean']:.2f}", va='center', fontsize=9)
    plt.show()

print(f'Loaded HIV matrix: {X.shape[0]} participants x {X.shape[1]} features; class counts HIV-={n0}, PLWH={n1}')
